# High Volume For-Hire Vehicle Data Analysis

## Data Acquisition
1. **Download Data**: Download the High Volume For-Hire Vehicle data for the entire year of 2022 for New York City. Access the data from [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). The data comprises 12 separate monthly files. A loop can be scripted in a notebook using the `requests` library or `wget` to automate the download process onto your Data Science VM.

## Data Analysis 
2. **Initial Data Loading**: Attempt to read the data for the entire year into a pandas dataframe.
    - **Note**: This process may lead to memory issues, observable by a crashing Jupyter kernel. Memory usage can be monitored using `htop` in a terminal within Jupyter Lab.
3. **Data Manipulation using pyspark**:
    - Add a new column, `day_of_week`, indicating the day of the week (Monday to Sunday) for each trip.
    - Compute the total number of trips for each day of the week.
    - Determine the average trip duration (in minutes) for each day of the week. (Does not require new column)
    - Calculate the average trip length (in miles) for each day of the week. (Does not require new column)

4. **Performance Measurement**:
    - Use the `%%time` cell magic to record the total time required for these operations, from reading the data file to producing the final output.

## Data Analysis with Dask (optional)
5. **Using Dask for Parallelization**:
    - Perform the analysis for the full year using Dask dataframes or another parallelization framework of your choice.
    - Record the total time required for this process as well.


Example of downloading one month

In [3]:
# ad Task 1

# %%bash
# mkdir -p ./downloaded-data

import subprocess

subprocess.run(["mkdir", "-p", "./downloaded-data"])

for i in range(1, 13):
    output = f"{i:02d}"
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2022-{output}.parquet"
    dest = f"./downloaded-data/hv_tripdata_2022-{output}.parquet"
    subprocess.run(["wget", url, "-O", dest])
    
    # wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2022-01.parquet -O ./downloaded-data/hv_tripdata_2022-01.parquet

--2026-05-15 20:55:45--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2022-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.181, 18.239.38.163, 18.239.38.83, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.181|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 374619188 (357M) [application/x-www-form-urlencoded]
Saving to: ‘./downloaded-data/hv_tripdata_2022-01.parquet’

     0K .......... .......... .......... .......... ..........  0% 8.16M 44s
    50K .......... .......... .......... .......... ..........  0%  206M 23s
   100K .......... .......... .......... .......... ..........  0% 8.43M 29s
   150K .......... .......... .......... .......... ..........  0%  236M 22s
   200K .......... .......... .......... .......... ..........  0% 12.8M 23s
   250K .......... .......... .......... .......... ..........  0%  114M 20s
   300K .......... .......... ....

....... .......... ..........  7% 17.6M 5s
 26500K .......... .......... .......... .......... ..........  7%  353M 5s
 26550K .......... .......... .......... .......... ..........  7%  353M 5s
 26600K .......... .......... .......... .......... ..........  7%  348M 5s
 26650K .......... .......... .......... .......... ..........  7% 3.96M 5s
 26700K .......... .......... .......... .......... ..........  7%  278M 5s
 26750K .......... .......... .......... .......... ..........  7%  326M 5s
 26800K .......... .......... .......... .......... ..........  7%  347M 5s
 26850K .......... .......... .......... .......... ..........  7%  357M 5s
 26900K .......... .......... .......... .......... ..........  7%  269M 5s
 26950K .......... .......... .......... .......... ..........  7%  353M 5s
 27000K .......... .......... .......... .......... ..........  7% 4.47M 6s
 27050K .......... .......... .......... .......... ..........  7% 4.28M 6s
 27100K .......... .......... .......... ....

In [ ]:
# ad Task 2

import pandas as pd
import glob

# Get all parquet files
files = sorted(glob.glob("./downloaded-data/hv_tripdata_2022-*.parquet"))

# Load and concatenate
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

print(df.shape)
print(df.dtypes)

# output:
# The Kernel crashed while executing code in the current cell or a previous cell. 
# Please review the code in the cell(s) to identify a possible cause of the failure. 
# Click here for more info. 
# View Jupyter log for further details.


: 

In [1]:
# ad Task 3

from pyspark.sql import SparkSession
from pyspark.sql.functions import dayofweek, date_format, col, avg, count, round
from pyspark.sql.functions import (unix_timestamp)

# Start Spark session
spark = SparkSession.builder \
    .appName("FHVHV 2022 Analysis") \
    .getOrCreate()

# Load all 12 parquet files at once
df = spark.read.parquet("./downloaded-data/hv_tripdata_2022-*.parquet")

# Add day_of_week column (Monday, Tuesday, etc.)
df = df.withColumn("day_of_week", date_format(col("pickup_datetime"), "EEEE"))

# Total trips per day of week
trips_per_day = df.groupBy("day_of_week") \
    .agg(count("*").alias("total_trips")) \
    .orderBy("total_trips", ascending=False)

# Average trip duration in minutes per day of week
avg_duration = df.groupBy("day_of_week") \
    .agg(
        round(
            avg((unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60), 2
        ).alias("avg_duration_minutes")
    )

# Average trip length in miles per day of week
avg_length = df.groupBy("day_of_week") \
    .agg(round(avg("trip_miles"), 2).alias("avg_trip_miles"))

# Show results
trips_per_day.show()
avg_duration.show()
avg_length.show()

26/05/15 21:03:47 WARN Utils: Your hostname, codespaces-4701ab resolves to a loopback address: 127.0.0.1; using 10.0.0.171 instead (on interface eth0)
26/05/15 21:03:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 21:03:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-----------+-----------+
|day_of_week|total_trips|
+-----------+-----------+
|   Saturday|   36274990|
|     Friday|   33560384|
|     Sunday|   30648605|
|   Thursday|   30409124|
|  Wednesday|   28413710|
|    Tuesday|   27061986|
|     Monday|   26047284|
+-----------+-----------+



+-----------+--------------------+
|day_of_week|avg_duration_minutes|
+-----------+--------------------+
|  Wednesday|               20.04|
|    Tuesday|                19.6|
|     Friday|               20.14|
|   Thursday|               20.55|
|   Saturday|               18.61|
|     Monday|               18.94|
|     Sunday|               18.14|
+-----------+--------------------+



+-----------+--------------+
|day_of_week|avg_trip_miles|
+-----------+--------------+
|  Wednesday|          4.98|
|    Tuesday|          4.96|
|     Friday|          4.93|
|   Thursday|          5.04|
|   Saturday|           4.9|
|     Monday|          5.13|
|     Sunday|          5.31|
+-----------+--------------+



In [5]:
%%time

# ad Task 4

from pyspark.sql import SparkSession
from pyspark.sql.functions import date_format, col, avg, count, round, unix_timestamp

spark = SparkSession.builder \
    .appName("FHVHV 2022 Analysis") \
    .getOrCreate()

df = spark.read.parquet("./downloaded-data/hv_tripdata_2022-*.parquet")
df = df.withColumn("day_of_week", date_format(col("pickup_datetime"), "EEEE"))

trips_per_day = df.groupBy("day_of_week") \
    .agg(count("*").alias("total_trips")) \
    .orderBy("total_trips", ascending=False)

avg_duration = df.groupBy("day_of_week") \
    .agg(
        round(
            avg((unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60), 2
        ).alias("avg_duration_minutes")
    )

avg_length = df.groupBy("day_of_week") \
    .agg(round(avg("trip_miles"), 2).alias("avg_trip_miles"))

trips_per_day.show()
avg_duration.show()
avg_length.show()

+-----------+-----------+
|day_of_week|total_trips|
+-----------+-----------+
|   Saturday|   36274990|
|     Friday|   33560384|
|     Sunday|   30648605|
|   Thursday|   30409124|
|  Wednesday|   28413710|
|    Tuesday|   27061986|
|     Monday|   26047284|
+-----------+-----------+



+-----------+--------------------+
|day_of_week|avg_duration_minutes|
+-----------+--------------------+
|  Wednesday|               20.04|
|    Tuesday|                19.6|
|     Friday|               20.14|
|   Thursday|               20.55|
|   Saturday|               18.61|
|     Monday|               18.94|
|     Sunday|               18.14|
+-----------+--------------------+



+-----------+--------------+
|day_of_week|avg_trip_miles|
+-----------+--------------+
|  Wednesday|          4.98|
|    Tuesday|          4.96|
|     Friday|          4.93|
|   Thursday|          5.04|
|   Saturday|           4.9|
|     Monday|          5.13|
|     Sunday|          5.31|
+-----------+--------------+

CPU times: user 87 ms, sys: 21.9 ms, total: 109 ms
Wall time: 5min 23s


In [ ]:
# ad Task 5 (first consideration)

import time
import dask.dataframe as dd

start = time.time()

# Load all 12 parquet files lazily
df = dd.read_parquet("./downloaded-data/hv_tripdata_2022-*.parquet")

# Add day_of_week column
df["day_of_week"] = df["pickup_datetime"].dt.day_name()

# Add trip duration in minutes
df["trip_duration_minutes"] = (
    (df["dropoff_datetime"] - df["pickup_datetime"]).dt.seconds / 60
)

# Total trips per day of week
trips_per_day = df.groupby("day_of_week").size().compute()

# Average trip duration per day of week
avg_duration = df.groupby("day_of_week")["trip_duration_minutes"].mean().compute()

# Average trip length per day of week
avg_length = df.groupby("day_of_week")["trip_miles"].mean().compute()

end = time.time()

# Display results
print("Total Trips per Day of Week:")
print(trips_per_day.sort_values(ascending=False))

print("\nAverage Trip Duration (minutes) per Day of Week:")
print(avg_duration.round(2).sort_values())

print("\nAverage Trip Length (miles) per Day of Week:")
print(avg_length.round(2).sort_values())

print(f"\nTotal time: {end - start:.2f} seconds")

# output
# The Kernel crashed while executing code in the current cell or a previous cell. 
# Please review the code in the cell(s) to identify a possible cause of the failure. 
# Click here for more info. 
# View Jupyter log for further details.

# Reason: Dask is loading everything into memory even though it parallize


: 

In [ ]:
# ad Task 5 (second consideration - load only needed columns)

import time
import dask.dataframe as dd

start = time.time()

cols = ["pickup_datetime", "dropoff_datetime", "trip_miles"]
df = dd.read_parquet("./downloaded-data/hv_tripdata_2022-*.parquet", columns=cols)

df["day_of_week"] = df["pickup_datetime"].dt.day_name()
df["trip_duration_minutes"] = (
    (df["dropoff_datetime"] - df["pickup_datetime"]).dt.seconds / 60
)

trips_per_day = df.groupby("day_of_week").size().compute()
avg_duration = df.groupby("day_of_week")["trip_duration_minutes"].mean().compute()
avg_length = df.groupby("day_of_week")["trip_miles"].mean().compute()

end = time.time()

print("Total Trips per Day of Week:")
print(trips_per_day.sort_values(ascending=False))
print("\nAverage Trip Duration (minutes):")
print(avg_duration.round(2))
print("\nAverage Trip Length (miles):")
print(avg_length.round(2))
print(f"\nTotal time: {end - start:.2f} seconds")

# output
# The Kernel crashed while executing code in the current cell or a previous cell. 
# Please review the code in the cell(s) to identify a possible cause of the failure. 
# Click here for more info. 
# View Jupyter log for further details.

: 

In [1]:
# ad Task 5 (third consideration - load only needed columns and limit memory usage)

from dask.distributed import Client
import time
import dask.dataframe as dd

client = Client(memory_limit="4GB")  # adjust to your available RAM
print(client)

start = time.time()

cols = ["pickup_datetime", "dropoff_datetime", "trip_miles"]
df = dd.read_parquet("./downloaded-data/hv_tripdata_2022-*.parquet", columns=cols)

df["day_of_week"] = df["pickup_datetime"].dt.day_name()
df["trip_duration_minutes"] = (
    (df["dropoff_datetime"] - df["pickup_datetime"]).dt.seconds / 60
)

trips_per_day = df.groupby("day_of_week").size().compute()
avg_duration = df.groupby("day_of_week")["trip_duration_minutes"].mean().compute()
avg_length = df.groupby("day_of_week")["trip_miles"].mean().compute()

end = time.time()

print("Total Trips per Day of Week:")
print(trips_per_day.sort_values(ascending=False))
print("\nAverage Trip Duration (minutes):")
print(avg_duration.round(2))
print("\nAverage Trip Length (miles):")
print(avg_length.round(2))
print(f"\nTotal time: {end - start:.2f} seconds")

<Client: 'tcp://127.0.0.1:40517' processes=2 threads=2, memory=7.45 GiB>


2026-05-15 22:00:32,378 - distributed.nanny - WARNING - Restarting worker
2026-05-15 22:00:40,720 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:34315' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('read_parquet-fused-ba648cf38946cf6ce3c7e7491f5f6a63', 2)} (stimulus_id='handle-worker-cleanup-1778882440.7204611')
2026-05-15 22:00:40,740 - distributed.nanny - WARNING - Restarting worker
2026-05-15 22:00:56,383 - distributed.nanny - WARNING - Restarting worker
2026-05-15 22:01:03,973 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:37431' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('read_parquet-fused-ba648cf38946cf6ce3c7e7491f5f6a63', 2)} (stimulus_id='handle-worker-cleanup-1778882463.9737132')
2026-05-15 22:01:03,980 - distributed.nanny - WARNING - Restarting worker
2026-05-15 22:01:03,968 - distributed.worker - ERROR - Worker stream died during communica

KilledWorker: Attempted to run task ('read_parquet-fused-ba648cf38946cf6ce3c7e7491f5f6a63', 0) on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://127.0.0.1:34763. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [ ]:
# given cell in the original notebook

#from pyspark.sql import SparkSession

# Create a Spark session
#spark = SparkSession.builder \
#    .appName("Read Parquet Example") \
#   .getOrCreate()

26/05/15 20:57:25 WARN Utils: Your hostname, codespaces-4701ab resolves to a loopback address: 127.0.0.1; using 10.0.0.171 instead (on interface eth0)
26/05/15 20:57:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 20:57:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# given cell in the original notebook

# Path to your Parquet file
# parquet_file_path = './downloaded-data/hv_tripdata_2022-01.parquet'

# Read the Parquet file
# df = spark.read.parquet(parquet_file_path)

# Show the first few records from the DataFrame
# df.show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--